# 09 Model Evaluation — SENTINEL
Test-only evaluation. Expect PR-AUC ~0.988, prec@500 = 1.0. Confusion at 0.5 / MEDIUM / HIGH.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))


In [ ]:
import json, numpy as np
from scipy.sparse import load_npz
from sklearn.metrics import confusion_matrix, average_precision_score, roc_auc_score
from src.xgboost_model import load_model, predict_proba
from src import config


In [ ]:
Xte = load_npz(config.ARTIFACTS_DIR / 'X_test.npz').toarray()
yte = np.load(config.ARTIFACTS_DIR / 'y_test.npy')
model = load_model()
probs = predict_proba(model, Xte)
thr = json.loads((config.ARTIFACTS_DIR / 'thresholds.json').read_text())
print(thr)
print('PR-AUC', average_precision_score(yte, probs), 'ROC-AUC', roc_auc_score(yte, probs))


In [ ]:
# Confusion matrices at 0.5 / MEDIUM / HIGH
for name, t in [('0.5', 0.5), ('MEDIUM', thr['MEDIUM']), ('HIGH', thr['HIGH'])]:
    pred = (probs >= t).astype(int)
    print(name, 'thr=%.4f' % t, '\n', confusion_matrix(yte, pred))


In [ ]:
# precision@100/500
order = np.argsort(probs)[::-1]
for k in [100, 500]:
    print(f'prec@{k}', yte[order[:k]].mean())
